# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [27]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "04-14-2025"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
update_date = "06-16-2025"

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

ModuleNotFoundError: spec not found for the module 'utils'

## Read Metadata 

In [28]:
# Read metadata

os.chdir(saved)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

9697
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
2274


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
16776,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-04-14 15:21:54,1,25-006243-005,SRP503016,NaN,"MILK, BULK TANK",SRS24711993,False,NaN,USA
16777,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,2025-04-14 15:21:54,1,25-006243-005,SRP503016,NaN,",",SRS24711993,False,NaN,
16778,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-04-14 15:20:51,1,25-006243-002,SRP503016,NaN,"MILK, BULK TANK",SRS24711992,False,NaN,USA
16779,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,2025-04-14 15:20:51,1,25-006243-002,SRP503016,NaN,",",SRS24711992,False,NaN,
16780,SRR33124596,WGS,262.12,61927215,PRJNA1102327,SAMN47941262,Viral,21760157,USDA-NVSL,2025,...,2025-04-14 15:20:49,1,25-006243-001,SRP503016,NaN,"MILK, BULK TANK",SRS24711991,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19045,SRR33830446,WGS,148.77,95232771,PRJNA1102327,SAMN48895426,Viral,37952018,USDA-NVSL,2025,...,2025-06-04 14:24:56,1,25-015677-004,SRP503016,NaN,"MILK, BULK TANK",SRS25266464,False,NaN,USA
19046,SRR33830447,WGS,148.43,106145246,PRJNA1102327,SAMN48895425,Viral,42182759,USDA-NVSL,2025,...,2025-06-04 14:25:01,1,25-015677-003,SRP503016,NaN,"MILK, BULK TANK",SRS25266463,False,NaN,USA
19047,SRR33830448,WGS,148.30,110812426,PRJNA1102327,SAMN48895424,Viral,43738503,USDA-NVSL,2025,...,2025-06-04 14:24:55,1,25-015677-002,SRP503016,NaN,"MILK, BULK TANK",SRS25266462,False,NaN,USA
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-04 14:24:53,1,24-036379-001-tile,SRP503016,NaN,"MILK, BULK TANK",SRS25266461,False,NaN,USA


In [29]:
# Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [30]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR33124594        WGS      246.43  145473401  PRJNA1102327   
1     SRR33124594        WGS      246.43  145473401  PRJNA1102327   
2     SRR33124595        WGS      240.38  139928333  PRJNA1102327   
3     SRR33124595        WGS      240.38  139928333  PRJNA1102327   
4     SRR33124596        WGS      262.12   61927215  PRJNA1102327   
...           ...        ...         ...        ...           ...   
2269  SRR33830446        WGS      148.77   95232771  PRJNA1102327   
2270  SRR33830447        WGS      148.43  106145246  PRJNA1102327   
2271  SRR33830448        WGS      148.30  110812426  PRJNA1102327   
2272  SRR33830449        WGS      131.52   67358891  PRJNA1102327   
2273  SRR33830450        WGS      131.13   69602732  PRJNA1102327   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN47941264          Viral  50832578   USDA-NVSL            2025  ...   
1     SAMN4

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List


In [5]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(metadata["name_state"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

0       USA
1          
2       USA
3          
6       USA
       ... 
2266    USA
2267    USA
2268    USA
2272    USA
2273    USA
Name: name_state, Length: 1975, dtype: object
1975


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,USA
1,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,
2,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,USA
3,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,
6,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,2025,...,SRR33124597,2025-05-09_10-52-41,SRR33124597.fa,B3.13,"NP:am8, PA:ea1, MP:ea1, PB1:am4, PB2:am2.2, NA...","am8:23-032005-001:NP, ea1:22-003707-003:PA, ea...","99.00%, 98.75%, 98.78%, 99.38%, 98.64%, 98.86%...","15, 27, 12, 14, 31, 16, 7, 29",Ran on FASTA - No Coverage Report,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA
2267,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA
2268,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA
2272,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA


In [6]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [7]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank.csv")
# os.chdir(temp_files)

# Get only updated dates

unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have
known_dates = metadata[(metadata["Collection_Date"] != "2024") & (metadata["Collection_Date"] != "2025")] # Dates we've already gotten

# Get new dates also 
# new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
unknown_dates["Collection_Date"] = updated_unknown_dates

metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata[["Collection_Date_Specific"]])

display(metadata)

https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_685060e08a0d775bfc0f318e&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_685060e3f2c337a6e409a400&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_685060e5f40edc29210fb4f1&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_685060e8a4dd2c26630c3a02&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809
2025-02-18
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
Unable to find collection date.
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=nuccore&query_key=1&WebEnv=MCID_685060f0954e9785950883aa&

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_17352\1076642909.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unknown_dates["Collection_Date"] = updated_unknown_dates


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Geo_Location
0,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,USA
1,SRR33124594,WGS,246.43,145473401,PRJNA1102327,SAMN47941264,Viral,50832578,USDA-NVSL,2025,...,SRR33124594,2025-05-09_10-46-09,SRR33124594.fa,B3.13,"NS:am1.1, NP:am8, NA:ea1, MP:ea1, PA:ea1, PB1:...","am1.1:22-010085-001:NS, am8:23-032005-001:NP, ...","99.17%, 98.66%, 98.86%, 98.78%, 98.70%, 99.43%...","7, 20, 16, 12, 28, 13, 28, 33",Ran on FASTA - No Coverage Report,
2,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,USA
3,SRR33124595,WGS,240.38,139928333,PRJNA1102327,SAMN47941263,Viral,48295907,USDA-NVSL,2025,...,SRR33124595,2025-05-09_10-52-41,SRR33124595.fa,B3.13,"HA:ea1, NP:am8, NS:am1.1, PB2:am2.2, PB1:am4, ...","ea1:22-003707-003:HA, am8:23-032005-001:NP, am...","98.36%, 98.66%, 99.17%, 98.55%, 99.38%, 98.86%...","28, 20, 7, 33, 14, 16, 12, 28",Ran on FASTA - No Coverage Report,
6,SRR33124597,WGS,264.06,73438935,PRJNA1102327,SAMN47941261,Viral,26040226,USDA-NVSL,2025,...,SRR33124597,2025-05-09_10-52-41,SRR33124597.fa,B3.13,"NP:am8, PA:ea1, MP:ea1, PB1:am4, PB2:am2.2, NA...","am8:23-032005-001:NP, ea1:22-003707-003:PA, ea...","99.00%, 98.75%, 98.78%, 99.38%, 98.64%, 98.86%...","15, 27, 12, 14, 31, 16, 7, 29",Ran on FASTA - No Coverage Report,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2266,SRR33830443,WGS,148.11,100597029,PRJNA1102327,SAMN48895429,Viral,39710063,USDA-NVSL,2025,...,SRR33830443,2025-06-06_07-56-21,SRR33830443.fa,B3.13,"NA:ea1, PB1:am4, MP:ea1, PA:ea1, NP:am8, HA:ea...","ea1:22-003707-003:NA, am4:23-001855-001:PB1, e...","98.72%, 99.52%, 98.98%, 98.88%, 98.86%, 98.06%...","18, 11, 10, 24, 17, 33, 38, 9",Ran on FASTA - No Coverage Report,USA
2267,SRR33830444,WGS,148.50,131987248,PRJNA1102327,SAMN48895428,Viral,51617990,USDA-NVSL,2025,...,SRR33830444,2025-06-06_07-56-20,SRR33830444.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, MP:...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.88%, 98.65%, 98.20%, 98.93%, 98.06%, 98.98%...","24, 19, 41, 16, 33, 10, 12, 9",Ran on FASTA - No Coverage Report,USA
2268,SRR33830445,WGS,148.12,115677479,PRJNA1102327,SAMN48895427,Viral,45674405,USDA-NVSL,2025,...,SRR33830445,2025-06-06_07-56-21,SRR33830445.fa,B3.13,"NP:am8, HA:ea1, PA:ea1, NS:am1.1, PB1:am4, PB2...","am8:23-032005-001:NP, ea1:22-003707-003:HA, ea...","98.93%, 98.24%, 98.79%, 98.93%, 99.56%, 98.51%...","16, 30, 26, 9, 10, 34, 19, 11",Ran on FASTA - No Coverage Report,USA
2272,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,SRR33830449,2025-06-06_07-56-20,SRR33830449.fa,B3.13,"NP:am8, NA:ea1, PB1:am4, HA:ea1, NS:am1.1, MP:...","am8:23-032005-001:NP, ea1:22-003707-003:NA, am...","98.93%, 98.86%, 99.47%, 98.43%, 99.17%, 98.95%...","16, 16, 12, 12, 7, 10, 31, 24",Ran on FASTA - No Coverage Report,USA


In [9]:
# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [10]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['hooded merganser', 'common raven', 'fox', '-', 'partridge', 'barn owl', 'wood duck', 'owl', 'skunk', 'great black-backed gull', 'mallard x american black duck hybrid', 'cattle', 'sand crane', 'hawk', 'chukar', 'canada goose', 'american black duck', 'tundra swan', 'quail', 'raccoon', 'common grackle', 'trumpeter swan', 'flamingo', 'snowy egret', 'red-tailed hawk', 'turkey', "'", 'mallard', 'snowy owl', 'mink', 'common loon', 'barred owl', 'great blue heron', 'sanderling', 'goose', 'red-shouldered hawk', 'cat', 'black vulture', "bonaparte's gull", 'gull', 'greater scaup', 'western sandpiper', 'great horned owl', 'cackling goose', 'american coot', 'red fox', 'chicken', 'cougar', 'pet food', 'falcon', 'glaucous gull', 'swan', 'american crow', 'snow goose', 'peregrine', 'guineafowl', 'bufflehead', 'rock goose', 'vulture', 'duck', 'bald eagle', 'mute swan', 'blue jay', 'dunlin', 'herring gull', 'western gull', "cooper's hawk", 'turkey vulture', "ross's goose", 'red-breasted merganser', 'le

In [11]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [12]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    
    if collection_date != collection_date: # If nan
        metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata.loc[num, "Collection_Date"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata.loc[num, "Collection_Date"] = date

    metadata = metadata.dropna(thresh=2)

# Make names

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["isolate"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata[["ReleaseDate", 'create_date', 'Collection_Date']])

,ReleaseDate,create_date,Collection_Date
0,2025-04-14,2025-04-14 15:21:54,2025
1,2025-04-14,2025-04-14 15:21:54,2025
2,2025-04-14,2025-04-14 15:20:51,2025
3,2025-04-14,2025-04-14 15:20:51,2025
6,2025-04-14,2025-04-14 15:21:19,2025
...,...,...,...
2266,2025-06-06,2025-06-04 14:25:00,2025
2267,2025-06-06,2025-06-04 14:25:01,2025
2268,2025-06-06,2025-06-04 14:24:54,2025
2272,2025-06-06,2025-06-04 14:24:53,2024


In [13]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [14]:
print(metadata)

              Run Assay Type  AvgSpotLen        Bases    BioProject  \
0     SRR33124594        WGS      246.43  145473401.0  PRJNA1102327   
2     SRR33124595        WGS      240.38  139928333.0  PRJNA1102327   
6     SRR33124597        WGS      264.06   73438935.0  PRJNA1102327   
8     SRR33124598        WGS      255.94   76531946.0  PRJNA1102327   
10    SRR33124599        WGS      145.79   94820034.0  PRJNA1102327   
...           ...        ...         ...          ...           ...   
2266  SRR33830443        WGS      148.11  100597029.0  PRJNA1102327   
2267  SRR33830444        WGS      148.50  131987248.0  PRJNA1102327   
2268  SRR33830445        WGS      148.12  115677479.0  PRJNA1102327   
2272  SRR33830449        WGS      131.52   67358891.0  PRJNA1102327   
2273  SRR33830450        WGS      131.13   69602732.0  PRJNA1102327   

         BioSample BioSampleModel       Bytes Center Name Collection_Date  \
0     SAMN47941264          Viral  50832578.0   USDA-NVSL            2

## Make FASTA files

In [15]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [16]:
# print(fasta_files.keys())

In [17]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = originals + "complete/" + pair + "_andersen_updated_" + update_date + ".fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33682399|A/CHICKEN/USA/25-014787-002/2025|H5N1|USA|2025|avian|B3.6
>SRR33125022|A/CROW/USA/25-004431-001/2025|H5N1|USA|2025|avian|A3
>SRR33125049|A/MALLARD/USA/25-005882-009/2025|H5N1|USA|2025|avian|A3
>SRR33125051|A/BALD EAGLE/USA/25-006076-002/2025|H5N1|USA|2025|avian|A3
>SRR33125053|A/MALLARD/USA/25-005882-006/2025|H5N1|USA|2025|avian|A3
>SRR33125055|A/MALLARD/USA/25-005882-003/2025|H5N1|USA|2025|avian|A3
>SRR33125062|A/BALD EAGLE/USA/25-006076-001/2025|H5N1|USA|2025|avian|A3
>SRR33125127|A/

## De-Duplication

In [19]:
# De-duplication 

# Gisaid 
dfs_gisaid_list = []
for genotype in genotypes:

    gisaid = downloads + "GISAID/complete/" + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "_North_America/"

    # gisaid = downloads + "Cats/Datasets/GISAID/"

    os.chdir(gisaid)

    dfs_gisaid = create_dataframes(gisaid)
    dfs_gisaid_list.append(dfs_gisaid)

dfs_gisaid = {}
for df_gisaid in dfs_gisaid_list:
    dfs_gisaid = dfs_gisaid | df_gisaid
# dfs_gisaid2 = create_dataframes(gisaid2)

B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [20]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(originals + "complete/")

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
B3.13_HA
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
B3.13_MP
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Andersen/complete/
B3.13_NA
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu

In [21]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [22]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
defaultdict(<class 'list'>, {'A3_HA': [   isolate_partial                                        full_header  \
0       004431-001  >SRR33125022|A/CROW/USA/25-004431-001/2025|H5N...   
1       005882-009  >SRR33125049|A/MALLARD/USA/25-005882-009/2025|...   
2       006076-002  >SRR33125051|A/BALD EAGLE/USA/25-006076-002/20...   
3       005882-006  >SRR33125053|A/MALLARD/USA/25-005882-006/2025|...   
4       005882-003  >SRR33125055|A/MALLARD/USA/25-005882-003/2025|...   
5       006076-001  >SRR33125062|A/BALD EAGLE/USA/25-006076-001/20...   
6       004415-008  >SRR33125127|A/RED-TAILED HAWK/USA/25-004415-0...   
7       011423-001  >SRR33319053|A/GREAT HORNED OWL/USA/25-011423-...   
8       011667-003  >SRR33319074|A/AMERICAN CROW

In [23]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

32
32


In [24]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

18
22
   isolate_partial                                        full_header  \
0        IZ24_0787  >EPI_ISL_19871019|A/northern_pintail/USA/IZ24_...   
1        IZ24_0636  >EPI_ISL_19871011|A/northern_pintail/USA/IZ24_...   
2        IZ24_0570  >EPI_ISL_19871010|A/northern_pintail/USA/IZ24_...   
3        IZ24_0474  >EPI_ISL_19871007|A/northern_pintail/USA/IZ24_...   
4        IZ23_0849  >EPI_ISL_19871006|A/northern_pintail/USA/IZ23_...   
5        IZ23_0847  >EPI_ISL_19871005|A/northern_pintail/USA/IZ23_...   
6       014764-001  >EPI_ISL_19882428|A/hawk/USA/014764-001/2025|H...   
7       005028-001  >EPI_ISL_19870521|A/american_crow/USA/005028-0...   
8       005028-002  >EPI_ISL_19870514|A/barred_owl/USA/005028-002/...   
9       010464-001  >EPI_ISL_19870554|A/common_raven/USA/010464-00...   
10      010511-001  >EPI_ISL_19870553|A/red-tailed_hawk/USA/010511...   
11      010511-002  >EPI_ISL_19870552|A/red-tailed_hawk/USA/010511...   
12      011667-003  >EPI_ISL_19851174|A/ameri

In [25]:
# # If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)
#             full_dfs[key].append(dataframes[i])

## Create FASTA files combining Andersen and GISAID

In [26]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.6_HA
B3.6_MP
B3.6_NA
B3.6_NP
B3.6_NS
B3.6_PA
B3.6_PB1
B3.6_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
